# Explainable Inverse Design Analysis of Couplers

This notebook demonstrates how to use the `squadds.ml` module to analyze the `coupler-CapNInterdigitalTee-cap_matrix` dataset. 

We will:
1. Load the curated dataset.
2. Use **Explainable Boosting Machines (EBM)** to identify the key geometric parameters (features) that drive each capacitance (target).
3. Use **Symbolic Regression (PySR)** to derive analytical equations for these capacitances.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from squadds.ml.pipeline import SQuADDSAnalysisPipeline
from squadds.ml.utils import prepare_features_with_interactions

## 1. Load Data
Load the curated dataset containing geometry parameters and simulated capacitance values.

In [ ]:
# Load dataset
df = pd.read_csv("../coupler_capacitance_data.csv")
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Define Features and Targets
We identified the varying design parameters in the previous step.

In [ ]:
features = ['cap_gap', 'cap_width', 'finger_count', 'finger_length']
targets = ['bottom_to_bottom', 'bottom_to_ground', 'ground_to_ground', 
           'top_to_bottom', 'top_to_ground', 'top_to_top']

## 3. Run Analysis Pipeline
We run the analysis for each target. The pipeline will:
- Train an EBM to find important features and interactions.
- Feed those features to PySR to find analytical equations.

**Note:** For demonstration speed, we limit PySR iterations (`niterations=20`). Increase this value (e.g., to 100) for more accurate and robust equations.

In [ ]:
pipeline = SQuADDSAnalysisPipeline(random_state=42)

# Output dictionary to store results
results = pipeline.analyze(
    df,
    features,
    targets,
    test_size=0.1,
    # EBM params
    ebm_kwargs={'interactions': 5, 'outer_bags': 4, 'inner_bags': 0},
    # PySR params - keep low for demo, increase for production
    symbolic_kwargs={
        'niterations': 20, 
        'populations': 15, 
        'population_size': 33,
        'maxsize': 25,
    },
    feature_threshold=0.01
)

## 4. View Results
### Best Equations

In [ ]:
# Print all equations without truncation
for t, r in results.items():
    print(f"{t:<20} | R2: {r['symbolic_metrics']['r2']:.4f}")
    print(f"Eq: {r['best_equation']}")
    print("-" * 80)

### Parity Plots
Visualizing the agreement between the Symbolic Regression model predictions and the true data.

In [ ]:
def plot_parity(results, df, target_name):
    if target_name not in results:
        return
    
    res = results[target_name]
    sym_model = res['symbolic_model']
    
    # Get features
    base_feats = res['selected_features']
    inte_pairs = res['interaction_pairs']
    
    # Prepare X
    X_all = df[features]
    X_sym = prepare_features_with_interactions(X_all, base_feats, inte_pairs)
    
    # Predict
    y_pred = sym_model.predict(X_sym)
    y_true = df[target_name]
    
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.5, edgecolor='k')
    
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Identity')
    
    plt.xlabel(f"True {target_name}")
    plt.ylabel(f"Predicted {target_name}")
    plt.title(f"Parity Plot: {target_name}\nR2: {res['symbolic_metrics']['r2']:.4f}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Plot for top 2 targets
plot_parity(results, df, 'top_to_bottom')
plot_parity(results, df, 'bottom_to_bottom')